# 1. Integer solution via Pyomo

Lecture: *Week 5 - Mixed Integer Programming*, PDF page 8 (*Integer Solution via Pyomo*). Pages 5–7 show the
LP optimum and why rounding it does not work. This is the Week 03 integer LP again (Week 03 demo 5), as a review.

**The problem.** maximise $P = 2x + 3.5y$ subject to $3x + 2y \le 14$, $2x + 3y \le 12.5$, $-x + y \le 1$,
$x, y \ge 0$, and **$x, y$ whole numbers**. In the code, $x$ is `x[0]` and $y$ is `x[1]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** $x = 3$, $y = 2$, $P = 13$. Without the whole-number rule the optimum is $(1.9, 2.9)$ with
$P = 13.95$; the best rounded point is $(2, 2)$ with $P = 11$ (PDF pages 6–7).

*Original script: `5 Source Files/Python/intlinprog_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Two integer variables. `NonNegativeIntegers` means whole numbers $\ge 0$: a lower bound of 0 and no upper bound.

In [ ]:
# Define the integer decision variables x[0] (= x) and x[1] (= y): whole numbers >= 0, no upper bound
model.x = Var([0, 1], domain=NonNegativeIntegers)

Objective coefficients: $P = 2x + 3.5y$.

In [ ]:
# Define the objective function coefficients as a list
f = [2, 3.5]

The objective, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(f[i] * model.x[i] for i in [0, 1]), sense=maximize)

The three constraints, one per line.

In [ ]:
# Add the constraints to the model
model.c1 = Constraint(expr=3 * model.x[0] + 2 * model.x[1] <= 14)    # 3x + 2y <= 14
model.c2 = Constraint(expr=2 * model.x[0] + 3 * model.x[1] <= 12.5)  # 2x + 3y <= 12.5
model.c3 = Constraint(expr=-model.x[0] + model.x[1] <= 1)            # -x + y <= 1

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model with the data filled in: check it against the problem above.

In [ ]:
# Display the model
model.pprint()

Solve. Because the variables are integers, HiGHS uses branch and bound (it solves LPs and splits on fractional values). In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the values: `x[0] = 3`, `x[1] = 2`, the objective 13, and each constraint's left-hand side (`Body`) next to its limit.

In [ ]:
# Display the results
model.display()

## What do you notice?

Look at the constraints in the output of `display()`. Which of them are at their limit at $(3, 2)$?

<details><summary>Answer</summary>

None of them: $3 \cdot 3 + 2 \cdot 2 = 13 < 14$, $2 \cdot 3 + 3 \cdot 2 = 12 < 12.5$ and $-3 + 2 = -1 < 1$.
An LP optimum is always at a corner of the feasible region, where constraints meet; an integer optimum is a
lattice point inside it and need not touch any constraint. That is why rounding the LP corner $(1.9, 2.9)$
does not find it.
</details>